In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/NguyenAn05/pocket-r1.git'
REPO_DIR = '/tmp/pocket-r1'
EXP4_LORA_DIR = '/kaggle/input/pocket-r1-exp4-lora'
PRM_V1_DIR = '/kaggle/input/pocket-r1-deberta-prm-v1'
PRM_V2_DIR = '/kaggle/working/deberta_prm_v2'
OUT_LORA_DIR = '/kaggle/working/exp5_coevolution_lora'
MERGED_DIR = '/tmp/exp5_coevolution_merged'
OUT_REPORT_DIR = '/kaggle/working/exp5_coevolution'
os.makedirs(OUT_REPORT_DIR, exist_ok=True)

print(subprocess.check_output(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'], text=True))
print('Exp 4 LoRA + FP files:', os.listdir(EXP4_LORA_DIR))
print('DeBERTa PRM v1 files:', os.listdir(PRM_V1_DIR))

if os.path.exists(REPO_DIR):
    subprocess.check_call(['rm', '-rf', REPO_DIR])
subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'peft', 'bitsandbytes', 'accelerate', 'sentencepiece', 'vllm'])
print('Exp 5 Co-evolution Gen 2 environment ready!', flush=True)


In [ ]:
import os, subprocess, sys

# Step 1: Warm-start upgrade DeBERTa PRM v1 -> v2 using mined False Positives from Exp 4 + Replay Buffer
env_up = os.environ.copy()
env_up['PYTHONUNBUFFERED'] = '1'
subprocess.check_call([
    sys.executable, '-u', f'{REPO_DIR}/scripts/upgrade_deberta_prm.py',
    '--init-prm-dir', PRM_V1_DIR,
    '--false-positives-file', f'{EXP4_LORA_DIR}/mined_false_positives.jsonl',
    '--output-dir', PRM_V2_DIR,
    '--replay-clean', '1500',
    '--replay-corrupt', '1500',
    '--epochs', '2',
], cwd=REPO_DIR, env=env_up)
print('DeBERTa PRM v2 upgrade complete!', flush=True)


In [ ]:
import os, subprocess, sys

# Step 2: Dual-GPU GRPO Gen 2 (GPU 0: Qwen3-4B QLoRA from Exp 4, GPU 1: DeBERTa PRM v2 FP16, League Tier 3)
rl_env = os.environ.copy()
rl_env['PYTHONUNBUFFERED'] = '1'
subprocess.check_call([
    sys.executable, '-u', f'{REPO_DIR}/scripts/train_grpo.py',
    '--config', f'{REPO_DIR}/configs/rl/exp5_coevolution.yaml',
    '--init-lora-dir', EXP4_LORA_DIR,
    '--prm-model-dir', PRM_V2_DIR,
    '--output-dir', OUT_LORA_DIR,
    '--merged-dir', MERGED_DIR,
    '--num-steps', '80',
    '--group-size', '4',
    '--merge',
], cwd=REPO_DIR, env=rl_env)
print('Exp 5 GRPO Gen 2 training & LoRA merge complete!', flush=True)


In [ ]:
import subprocess, time, urllib.request, sys, os

LOG_PATH = '/tmp/vllm_exp5.log'
log_file = open(LOG_PATH, 'w')

vllm_cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MERGED_DIR,
    '--served-model-name', 'Qwen/Qwen3-4B-exp5-coevolution',
    '--dtype', 'half',
    '--tensor-parallel-size', '2',
    '--max-model-len', '2048',
    '--gpu-memory-utilization', '0.85',
    '--enforce-eager',
    '--host', '127.0.0.1',
    '--port', '8000',
]
print('Starting vLLM for Exp 5:', ' '.join(vllm_cmd), flush=True)
vllm_proc = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=subprocess.STDOUT)

ready = False
for sec in range(420):
    if vllm_proc.poll() is not None:
        log_file.flush()
        with open(LOG_PATH) as f:
            print('vLLM exited early! Last logs:\n', f.read()[-4000:])
        raise RuntimeError(f'vLLM crashed with code {vllm_proc.returncode}')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=2) as r:
            if r.status == 200:
                print(f'vLLM ready in {sec}s:', r.read().decode(), flush=True)
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    vllm_proc.terminate()
    raise TimeoutError('vLLM did not become ready within 420s')

try:
    eval_cmd = [
        sys.executable, f'{REPO_DIR}/scripts/run_exp0_baseline.py',
        '--config', f'{REPO_DIR}/configs/eval/exp0_baseline.yaml',
        '--backend', 'openai_compatible',
        '--model', 'Qwen/Qwen3-4B-exp5-coevolution',
        '--base-url', 'http://127.0.0.1:8000/v1',
        '--workers', '24',
        '--no-ev',
        '--experiment-name', 'exp5_coevolution',
        '--output-dir', OUT_REPORT_DIR,
    ]
    print('Evaluating Exp 5 on 2000 Held-out hands:', ' '.join(eval_cmd), flush=True)
    subprocess.check_call(eval_cmd, cwd=REPO_DIR)
finally:
    vllm_proc.terminate()
    try:
        vllm_proc.wait(timeout=20)
    except Exception:
        vllm_proc.kill()
    log_file.close()
    print('Exp 5 Co-evolution Gen 2 complete! Outputs:', os.listdir(OUT_REPORT_DIR), flush=True)
